# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # .to_json() is optional, metadata is an object
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review all available record sets, fields, and their `@id`s from the dataset schema.

In Croissant datasets, each `RecordSet` corresponds to a table-like entity (i.e., collection of records, e.g., a CSV or spreadsheet sheet), described by its unique `@id`.
Each `RecordSet` contains `Field`s, which map to columns, each also identified by their `@id`.

**Let's print a summary of all record sets, and for each, their fields.**

In [ ]:
from pprint import pprint

# List all record sets from the Croissant schema
if hasattr(dataset, 'record_sets'):
    record_sets = dataset.record_sets  # List of RecordSet objects
else:
    print("No record sets found in the metadata.")
    record_sets = []

record_set_ids = []
for rs in record_sets:
    print(f"RecordSet: {rs.name} | @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    Field: {field.name} | @id: {field.id} | DataType: {getattr(field, 'data_type', 'Unknown')}")
    print()
if not record_set_ids:
    print("No record sets available to display fields.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

We use the record set and field `@id`s as listed above to extract records.

If there are multiple record sets, we'll load them into a dictionary of DataFrames, keyed by their `@id`.

In [ ]:
# Extract all record sets into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet @id: {rs_id} | {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"Could not load records for RecordSet {rs_id}: {e}")

# See what was loaded
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in first loaded RecordSet (@id: {main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping.

- We'll select a numeric field (column) for analysis.
- We'll filter records with high values on that field.
- Normalize the values.
- Optionally, group by a categorical field.

**Make sure to use the `@id` of the chosen fields as the keys to reference the columns.**

In [ ]:
# For demonstration, try to pick one numeric and one category field from the first loaded record set
import numpy as np

# If no data, skip this cell
if dataframes:
    df = dataframes[main_rs_id]
    numeric_field_id = None
    category_field_id = None

    # Guess numeric and category fields by dtype (in real usage, use explicit @id)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) and numeric_field_id is None:
            numeric_field_id = col
        elif pd.api.types.is_object_dtype(df[col]) and category_field_id is None:
            category_field_id = col
    if numeric_field_id is None:
        print("No numeric field found.")
    else:
        print(f"Using numeric field: {numeric_field_id}")

        # Filtering rows with numeric_field > threshold (using 10, or minimum+1 if not appropriate)
        threshold = 10
        valid_rows = df[numeric_field_id].apply(lambda x: pd.notnull(x) and np.isfinite(x))
        possible_min = df.loc[valid_rows, numeric_field_id].min()
        if possible_min > threshold:
            threshold = possible_min

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (n={filtered_df.shape[0]}):")
        display(filtered_df.head())

        # Normalization
        mean_ = filtered_df[numeric_field_id].mean()
        std_ = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_) / std_
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if available
        if category_field_id is not None and category_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(category_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {category_field_id}:")
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

*Examples: histogram of the numeric field, or boxplot/grouped barplot against the category field.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Proceed only if EDA loaded data
if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if category_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=category_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {category_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:

- Load Croissant-compliant datasets with `mlcroissant`
- Examine metadata, record sets, and fields using their `@id` attributes
- Extract and analyze data tables defined in the schema using the correct identifiers
- Apply basic EDA and visualization

**Further steps:**
- Explore more complex operations (merging record sets, advanced filtering)
- Apply statistical modeling for in-depth analysis
- Integrate supplementary annotation or documentation fields if available in the Croissant schema.

See the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for more advanced examples and schema details.